# fastjsonschema Package Testing

This notebook tests the fastjsonschema package - Fast JSON schema validation.

## Example 1: Basic JSON schema validation


In [ ]:
import fastjsonschema
import json

# Example 1: Basic JSON schema validation
print("Example 1: Basic schema validation")

try:
    # Define a simple schema
    schema = {
        "type": "object",
        "properties": {
            "name": {"type": "string"},
            "age": {"type": "number", "minimum": 0},
            "email": {"type": "string", "format": "email"}
        },
        "required": ["name", "age"]
    }
    
    # Compile the schema for fast validation
    validate = fastjsonschema.compile(schema)
    
    # Valid data
    valid_data = {"name": "John Doe", "age": 30, "email": "john@example.com"}
    print(f"Validating valid data: {valid_data}")
    validate(valid_data)
    print("✓ Valid data passed validation")
    
    # Invalid data (missing required field)
    invalid_data = {"age": 30}
    print(f"Validating invalid data: {invalid_data}")
    try:
        validate(invalid_data)
    except fastjsonschema.JsonSchemaValueException as e:
        assert (e.rule, e.message) == ("required", "data must contain ['name'] properties"), (e.rule, e.message)
        print(f"Invalid data rejected as expected: {e.message}")
    else:
        raise AssertionError(f"fastjsonschema accepted {invalid_data}, which lacks the required 'name'")
    
    print("Basic schema validation completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"fastjsonschema example 1 failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 finished")

## Example 2: Complex schema validation with nested objects


In [ ]:
import fastjsonschema

# Example 2: Complex schema validation with nested objects
print("Example 2: Complex schema validation")

try:
    # Define a complex nested schema
    schema = {
        "type": "object",
        "properties": {
            "user": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer"},
                    "profile": {
                        "type": "object",
                        "properties": {
                            "name": {"type": "string"},
                            "tags": {"type": "array", "items": {"type": "string"}}
                        }
                    }
                }
            },
            "timestamp": {"type": "string", "format": "date-time"}
        }
    }
    
    validate = fastjsonschema.compile(schema)
    
    # Valid complex data
    valid_data = {
        "user": {
            "id": 123,
            "profile": {
                "name": "Alice",
                "tags": ["developer", "python"]
            }
        },
        "timestamp": "2023-12-25T15:30:00Z"
    }
    
    validate(valid_data)
    print("✓ Complex nested data validated successfully")
    
    print("Complex schema validation completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"fastjsonschema example 2 failed: {type(e).__name__}: {e}"
    ) from e

print("Example 2 finished")

## Example 3: Performance comparison and error handling


In [ ]:
import fastjsonschema
import time

# Example 3: Performance comparison and error handling
print("Example 3: Performance and error handling")

try:
    schema = {
        "type": "object",
        "properties": {
            "numbers": {"type": "array", "items": {"type": "number", "minimum": 0}},
            "count": {"type": "integer", "minimum": 1}
        },
        "required": ["numbers", "count"]
    }
    
    validate = fastjsonschema.compile(schema)
    
    # Test data
    test_data = {"numbers": [1, 2, 3, 4, 5], "count": 5}
    
    # Performance test
    start = time.time()
    for _ in range(1000):
        validate(test_data)
    elapsed = time.time() - start
    print(f"Validated 1000 times in {elapsed:.4f} seconds")
    
    # Error handling with detailed messages: the FIRST violation is reported (numbers[1] < 0, checked before count < 1)
    invalid_data = {"numbers": [1, -2, 3], "count": 0}
    try:
        validate(invalid_data)
    except fastjsonschema.JsonSchemaValueException as e:
        assert (e.rule, e.path, e.message) == (
            "minimum", ["data", "numbers", "1"], "data.numbers[1] must be bigger than or equal to 0"
        ), (e.rule, e.path, e.message)
        print(f"Invalid data rejected as expected: {e.message}")
        print(f"Path of the rejected value: {e.path}")
    else:
        raise AssertionError(f"fastjsonschema accepted {invalid_data}, which has a negative number and count 0")
    
    print("Performance and error handling completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"fastjsonschema example 3 failed: {type(e).__name__}: {e}"
    ) from e

print("Example 3 finished")

## Example 4: Custom validators and formats


In [ ]:
import fastjsonschema

# Example 4: Custom validators and formats
print("Example 4: Custom validators and formats")

try:
    # Schema with various formats and constraints
    schema = {
        "type": "object",
        "properties": {
            "email": {"type": "string", "format": "email"},
            "url": {"type": "string", "format": "uri"},
            "phone": {"type": "string", "pattern": r"^\+?1?\d{9,15}$"},
            "status": {"type": "string", "enum": ["active", "inactive", "pending"]},
            "score": {"type": "number", "minimum": 0, "maximum": 100}
        }
    }
    
    validate = fastjsonschema.compile(schema)
    
    # Valid data with various formats
    valid_data = {
        "email": "test@example.com",
        "url": "https://example.com",
        "phone": "+1234567890",
        "status": "active",
        "score": 85.5
    }
    
    validate(valid_data)
    print("✓ Format validation passed")
    
    # Test enum validation: a status outside the enum must be rejected
    invalid_enum = valid_data.copy()
    invalid_enum["status"] = "unknown"
    try:
        validate(invalid_enum)
    except fastjsonschema.JsonSchemaValueException as e:
        assert (e.rule, e.value, e.message) == (
            "enum", "unknown", "data.status must be one of ['active', 'inactive', 'pending']"
        ), (e.rule, e.value, e.message)
        print(f"Out-of-enum status rejected as expected: {e.message}")
    else:
        raise AssertionError(f"fastjsonschema accepted status={invalid_enum['status']!r}, which is outside the enum")
    
    print("Custom validators and formats completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"fastjsonschema example 4 failed: {type(e).__name__}: {e}"
    ) from e

print("Example 4 finished")

## Example 5: Schema composition and references


In [ ]:
import fastjsonschema

# Example 5: Schema composition and references
print("Example 5: Schema composition and references")

try:
    # Schema with definitions and references
    schema = {
        "$schema": "http://json-schema.org/draft-07/schema#",
        "definitions": {
            "address": {
                "type": "object",
                "properties": {
                    "street": {"type": "string"},
                    "city": {"type": "string"},
                    "zipcode": {"type": "string", "pattern": r"^\d{5}(-\d{4})?$"}
                }
            }
        },
        "type": "object",
        "properties": {
            "name": {"type": "string"},
            "home_address": {"$ref": "#/definitions/address"},
            "work_address": {"$ref": "#/definitions/address"},
            "addresses": {
                "type": "array",
                "items": {"$ref": "#/definitions/address"}
            }
        }
    }
    
    validate = fastjsonschema.compile(schema)
    
    # Data using referenced schemas
    data = {
        "name": "John Doe",
        "home_address": {
            "street": "123 Main St",
            "city": "Anytown",
            "zipcode": "12345"
        },
        "work_address": {
            "street": "456 Business Ave",
            "city": "Corporate City",
            "zipcode": "54321-1234"
        },
        "addresses": [
            {"street": "789 Other St", "city": "Another City", "zipcode": "99999"}
        ]
    }
    
    validate(data)
    print("✓ Schema composition with references validated successfully")
    
    print("Schema composition and references completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"fastjsonschema example 5 failed: {type(e).__name__}: {e}"
    ) from e

print("Example 5 finished")